# Ü4.1 — DynamicFrames im Notebook erkunden (LÖSUNG)

Interaktive Glue-Session. Ablauf: `raw.orders` lesen → ApplyMapping → Filter →
`toDF()` + Spark SQL → Parquet schreiben → zählen.

**Voraussetzung:** Crawler aus Ü3.1 hat `raw.orders` katalogisiert.
Ausgabe landet unter `s3://gfu-glue-training-629452195361/processed/orders_nb/`
(eigener Prefix, kollidiert nicht mit dem Job-Output aus Ü5.1).

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping, Filter
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

## 1) Lesen — DynamicFrame aus dem Katalog

In [ ]:
orders = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="orders", transformation_ctx="orders"
)
orders.printSchema()
orders.toDF().show(truncate=False)

## 2) ApplyMapping — Spalten bereinigen

In [ ]:
# Schmutzige Spaltennamen (mit Leerzeichen) bereinigen + Typen setzen.
mapped = ApplyMapping.apply(
    frame=orders,
    mappings=[
        ("cust id", "string", "customer_id", "string"),
        ("order total", "string", "order_total", "double"),
        ("order date", "string", "order_date", "string"),
        ("status", "string", "status", "string"),
    ],
)
mapped.printSchema()

## 3) Filter — nur versendete Bestellungen

In [ ]:
# Nur exakt "shipped": Zeile C002 ("shipped, partial") faellt bewusst raus —
# gutes Gespraech ueber exakte vs. unscharfe Filter.
shipped = Filter.apply(frame=mapped, f=lambda row: row["status"] == "shipped")
print("shipped rows:", shipped.count())

## 4) toDF + Spark SQL — Aggregation

In [ ]:
# DynamicFrame -> DataFrame, dann Spark SQL fuer eine Aggregation.
df = mapped.toDF()
df.createOrReplaceTempView("orders")
spark.sql(
    "SELECT status, count(*) AS n, round(sum(order_total), 2) AS total "
    "FROM orders GROUP BY status ORDER BY n DESC"
).show(truncate=False)

## 5) Parquet schreiben

In [ ]:
# Als Parquet in einen eigenen Notebook-Prefix schreiben.
OUTPUT = "s3://gfu-glue-training-629452195361/processed/orders_nb/"
mapped.toDF().write.mode("overwrite").parquet(OUTPUT)
print("geschrieben nach", OUTPUT)

## 6) Zählen

In [ ]:
print("Zeilen gesamt (mapped):", mapped.count())